<a href="https://colab.research.google.com/github/ryanconquers/FlyrankAi_MLintern/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Refresh / Content Opportunity Scoring
I'm picking this lane because I've already built the core intuition for it in Notebooks 01 and 02 — ranking pages by a transparent hand-rule, then watching a learned model beat it (baseline Precision@50 = 0.24 vs. random forest = 0.74 on the starter slice). The underlying problem — rank a large candidate pool by a weighted blend of signals so a reviewer with limited capacity works the most promising cases first — is structurally the same pattern I use in my own equity research work (a scoring system that ranks companies for an analyst to review), so this lane lets me deepen a skill I'm already applying elsewhere. On the eligible slice, 54.2% of pages are currently declining — a large signal — while only 0.1% (17 pages) qualify as strictly stale-but-visible under the starter's thresholds. The declining share is the stronger justification for this lane; the small stale-visible count suggests that particular reason code may need loosening, or that the real opportunity concentrates elsewhere in the funnel rather than in strict staleness.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

The decision: Which content pages should a reviewer prioritize for action, out of a much larger pool than anyone has time to check by hand.
Who acts on it: A content/SEO reviewer (or account strategist) with limited weekly capacity — they can realistically work through maybe 20-50 flagged pages, not thousands. My output is a ranked queue with reason codes, so they know not just which pages but why each one surfaced, and can pick a realistic action per page: refresh, expand, protect, prune, or monitor.
Cost of a wrong call, and it's asymmetric:

False positive (a page gets flagged and reviewed, but wasn't actually worth the time): a wasted hour or two of reviewer capacity. Annoying, and it compounds if it happens often since capacity is the truly scarce resource — but each individual miss is cheap.
False negative (a genuinely declining, high-demand page never makes the queue): this is the expensive error. Nobody looks at it, so the decline continues silently and compounds — by the time it's obvious without the model, it's already lost more visibility/traffic than it needed to.

Because false negatives cost more than false positives here, I'll weight validation toward recall on high-volume pages even if that means accepting somewhat lower precision — better to over-flag a borderline page than silently miss a real one.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
 import pandas as pd
 import os, subprocess

if not os.path.exists("flyrank-ml-internship-starter"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)

os.chdir("flyrank-ml-internship-starter")
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV still not found"
print("Data found, ready to go.")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. How big is the eligible population? (matches starter's own filter)
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
print(f"Eligible pages: {len(eligible):,} of {len(df):,} total "
      f"({len(eligible)/len(df):.1%})")

# 2. How much of the "problem" actually exists in this slice?
declining_pct = (eligible["trend_direction"] == "down").mean()
print(f"Share currently declining: {declining_pct:.1%}")

# 3. How big is the "stale but still visible" opportunity bucket?
stale_visible = eligible[
    (eligible["days_since_last_update"] >= 180) &
    (eligible["impressions_90d"] >= 500)
]
print(f"Stale-but-visible pages: {len(stale_visible):,} "
      f"({len(stale_visible)/len(eligible):.1%} of eligible)")

Working dir: /content/flyrank-ml-internship-starter
Data found, ready to go.
Eligible pages: 30,000 of 30,000 total (100.0%)
Share currently declining: 54.2%
Stale-but-visible pages: 17 (0.1% of eligible)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What my work can say:

What patterns are observed in the historical data — e.g. "pages with X characteristic tended to show Y outcome over the sample window."
What's directional — a signal points toward risk or opportunity, not a certainty.
What's useful for decision-support — a ranked queue that helps a reviewer spend limited time on the most promising candidates first, backed by reason codes they can inspect and override.
Whether my model beats a transparent baseline rule, measured honestly with client-holdout validation (not in-sample scoring, which overstates performance).

What my work will never say:

That a refresh or content change caused a recovery — that requires an actual experiment (e.g. an A/B test on the pages themselves), which this data doesn't give me. I can say a page was flagged and later recovered; I can't say the flag or the fix caused it.
That I've discovered or predicted anything about Google's actual ranking algorithm. I only have observable, after-the-fact signals (impressions, clicks, position, engagement) — not the algorithm's inputs or logic.
That a "declining" label is a guarantee of anything — it could be consolidation (a sibling page absorbing demand), seasonality, or plain noise, not a real, sustained decline, unless I've specifically ruled those out.
Anything from AI-referral data beyond cautious pattern description — those rows are too sparse (30,177 out of 78.8M) to support any confident claim, let alone a classifier.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.